# DPO v2 — preference alignment on top of SFT v2

The DPO stage for the **v2 pipeline**: it starts from the SFT v2 model (redacted data, loss on the final reply only) and learns from hard-negative pairs built from **the same redacted data**. v1's notebook (`dpo_training.ipynb`, run v7) and its files in `MyDrive/DPO/` are kept unchanged.

## What changed from v8, and why

| v8 | v2 | Why |
|---|---|---|
| Starts from SFT v1 | Starts from **SFT v2**, and checks it really is v2 (`loss_on = final_reply_only`) | The v2 pipeline must not mix in a v1 model |
| Pairs from the unredacted data | Pairs from the **redacted** build, refused otherwise | The names problem |
| Nothing ties the pairs to the SFT model | The pairs' manifest must show **the same `train.jsonl` and `validation.jsonl` SHA-256** as SFT v2's run summary | DPO's "unseen" test conversations must be unseen by *this* SFT model |
| `rpo_alpha` anchor; falls back to plain DPO if missing | Anchor written in today's TRL API (`loss_type=["sigmoid","sft"]`, `loss_weights=[1, 0.5]`), `rpo_alpha` on older TRL, and **refuses** plain DPO | TRL 1.15 removed `rpo_alpha` and `max_prompt_length`: v8 would quietly lose the anchor, then crash |
| Prompts pre-templated by the builder with the original NileChat tokenizer | Pairs store **messages**; the notebook applies the SFT v2 model's own chat template | DPO prompts match SFT v2's training prompts exactly |
| Reply ending assumed or patched | The tokenized data is **checked**: every reply ends with exactly one `<\|im_end\|>`, as in SFT v2's labels | TRL adds the tokenizer's EOS, which may be a different token |
| Over-long pairs truncated by TRL (`keep_start` cuts the reply) | Over-long pairs **dropped and counted** before training | The reply is what DPO compares |
| Dev and test from the same file and seed (500 test pairs → 300 after overlap) | Split **by conversation**; test = all remaining validation conversations (larger, so smaller differences become measurable) | A clean, bigger test set |
| Resumes from any checkpoint in the folder | Resumes **only** if config, pairs and base model match `run_config.json` (as in SFT v2) | A run is never a mix of two settings |
| Head-to-head loads any other run's adapter into this model | Only runs trained on **the same SFT model**; v1 runs are compared in the evaluation notebook | A v7 adapter on the SFT v2 base would give meaningless numbers |
| Real conversations and generations in `run_summary.json` | Totals in `run_summary.json`; real text only in `generations_private.json`; names in generations **counted** | Summaries get shared; generations can contain real text |

## Before you start
1. On your machine, build the pairs from the redacted data:
   `python scripts/build_dpo_pairs_v2.py --data-dir ../data-preparation/outputs/redacted --output-dir dpo_v2_pairs --private-names ~/private_names.txt`
2. Upload `train.jsonl`, `dev.jsonl`, `test.jsonl` and `manifest.json` from `dpo_v2_pairs/` to `MyDrive/ahmed_twin/dpo_v2/pairs/`.
3. `MyDrive/ahmed_twin/model_merged_nilechat_v2/` must exist (SFT v2, with its `sft_run_summary.json`), and `MyDrive/ahmed_twin/private_names.txt`.
4. Runtime → T4 GPU → Run all. After a disconnect: Run all again (it resumes).

**Before committing: Edit → Clear all outputs.**

## A. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
assert torch.cuda.is_available(), "No GPU -- Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install unsloth  # brings a TRL version Unsloth supports; do not upgrade TRL separately

`UNSLOTH_RETURN_LOGITS=1` makes the model return real logits, which the preference scoring needs. Library versions are saved with the run.

In [ ]:
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

import unsloth  # must come before trl / transformers / peft
from unsloth import FastLanguageModel, PatchDPOTrainer
PatchDPOTrainer()

import gc
import hashlib
import inspect
import json
import re
import shutil
import statistics
from collections import Counter
from math import comb

import datasets
import peft
import transformers
import trl
from datasets import Dataset
from transformers import set_seed

LIBRARY_VERSIONS = {"unsloth": getattr(unsloth, "__version__", "unknown"), "torch": torch.__version__,
                    "transformers": transformers.__version__, "trl": trl.__version__, "peft": peft.__version__,
                    "datasets": datasets.__version__}
print(LIBRARY_VERSIONS)

### A1. Configuration: everything you may change

`CONFIG` is saved with the run (`run_config.json`). Resuming with a different `CONFIG`, different pairs or a different SFT model is refused: use a new `RUN_NAME`.

| If the diagnosis says | Change |
|---|---|
| Preference accuracy barely moved | `beta` 0.2 → 0.1 (new run) |
| `logps/chosen` falling more than 5% | `sft_weight` 0.5 → 1.0 |
| Reward accuracy ≥ 98% early | Pairs too easy: rebuild with `--max-similarity 0.4` |
| Best checkpoint is the first evaluation | `learning_rate` 1e-5 → 5e-6 |

In [ ]:
RUN_NAME = "dpo_v2_beta02_sft05"

DRIVE_ROOT = "/content/drive/MyDrive"
PROJECT_DIR = f"{DRIVE_ROOT}/ahmed_twin"
SFT_MODEL_PATH = f"{PROJECT_DIR}/model_merged_nilechat_v2"
PAIRS_DIR = f"{PROJECT_DIR}/dpo_v2/pairs"
RUNS_DIR = f"{PROJECT_DIR}/dpo_v2/runs"
PRIVATE_NAMES_PATH = f"{PROJECT_DIR}/private_names.txt"

RUN_DIR = f"{RUNS_DIR}/{RUN_NAME}"
CHECKPOINT_DIR = f"{RUN_DIR}/checkpoints"
ADAPTER_DIR = f"{RUN_DIR}/adapter"
MERGED_DIR = f"{RUN_DIR}/merged"
RUN_CONFIG_PATH = f"{RUN_DIR}/run_config.json"
RUN_SUMMARY_PATH = f"{RUN_DIR}/run_summary.json"
BASELINE_PATH = f"{RUN_DIR}/baseline.json"                 # SFT v2 scores, reused when resuming (private text inside)
GENERATIONS_PATH = f"{RUN_DIR}/generations_private.json"   # real text: never share
assert "/DPO/" not in RUN_DIR, "v2 runs must not go into v1's DPO folder"

CONFIG = {
    "loss": "sigmoid DPO + sft anchor on chosen",
    "beta": 0.2,                  # v7's value: the selected v1 run
    "sft_weight": 0.5,            # v7's rpo_alpha: keeps Ahmed's real replies likely
    "max_length": 1024,           # prompt + reply tokens; longer pairs are dropped, never truncated
    "lora_r": 32,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "learning_rate": 1e-5,
    "lr_scheduler_type": "linear",
    "warmup_steps": 10,
    "weight_decay": 0.01,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "per_device_eval_batch_size": 1,
    "num_train_epochs": 3,        # a ceiling: early stopping usually ends sooner
    "eval_steps": 25,
    "early_stopping_patience": 3,
    "seed": 0,
}

ALLOW_PLAIN_DPO = False       # True only to run DPO without the anchor on purpose
REQUIRE_NAME_CHECK = True     # refuse to train if private_names.txt is missing
MAX_DROPPED_FRACTION = 0.02   # refuse if more than 2% of pairs are over max_length
MIN_FREE_DRIVE_GB = 3         # checkpoints (~0.4 GB each, 2-3 kept) + the adapter
MIN_RELATIVE_LORA_CHANGE = 1e-6
MIN_ADAPTER_BYTES = 50e6      # a rank-32 adapter for a 3B model is ~240 MB; less means a failed save
TEST_PAIRS_LIMIT = None       # None = score every test pair (slower, most reliable)
N_HELDOUT_GENERATIONS = 5
HANDWRITTEN_TEST_MESSAGES = [
    "ممكن تساعدني في حاجة مهمة النهاردة؟ محتاج رأيك بجد",
    "معلش تأخرت في الرد كنت مشغول شوية",
    "ايه رأيك في اللي حصل النهاردة في الشغل؟",
    "مش عارف اعمل ايه في القرار ده ممكن تنصحني",
    "بجد شكرا انك كنت جنبي في الموضوع ده",
]
COMPARE_WITH_RUNS = []        # other dpo_v2 run names, compared only if trained on the same SFT model
MERGE_AFTER_TRAINING = False  # True writes a standalone merged model (~6-7 GB) to MERGED_DIR

set_seed(CONFIG["seed"])
print("Run folder:", RUN_DIR)

## B. Inputs: the SFT v2 model and the pairs must belong together
### B1. The base model is SFT v2

In [ ]:
def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


SFT_SUMMARY_PATH = os.path.join(SFT_MODEL_PATH, "sft_run_summary.json")
assert os.path.exists(SFT_SUMMARY_PATH), f"Missing {SFT_SUMMARY_PATH} -- is {SFT_MODEL_PATH} the merged SFT v2 model?"
with open(SFT_SUMMARY_PATH, encoding="utf-8") as f:
    sft_summary = json.load(f)
if sft_summary.get("config", {}).get("loss_on") != "final_reply_only":
    raise ValueError("This is not an SFT v2 model (its summary does not say loss_on = final_reply_only).")

BASE_IDENTITY = {
    "sft_model_path": SFT_MODEL_PATH,
    "sft_summary_sha256": sha256_of(SFT_SUMMARY_PATH),
    "sft_best_checkpoint": sft_summary.get("best_checkpoint"),
    "sft_best_eval_loss": sft_summary.get("best_eval_loss"),
}
print(f"SFT v2: best eval loss {sft_summary['best_eval_loss']:.4f} at "
      f"{os.path.basename(str(sft_summary.get('best_checkpoint')))}")

### B2. The pairs were built from the same data, and are clean

The manifest written by `build_dpo_pairs_v2.py` must show the **same SHA-256** for `train.jsonl` and `validation.jsonl` as SFT v2's run summary. That proves the DPO test conversations were never seen by this SFT model, and that the data was redacted. Then the files are checked against the manifest, dev and test against train, and every text for private names (counts only).

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


paths = {name: f"{PAIRS_DIR}/{name}.jsonl" for name in ("train", "dev", "test")}
MANIFEST_PATH = f"{PAIRS_DIR}/manifest.json"
for path in [*paths.values(), MANIFEST_PATH]:
    assert os.path.exists(path), f"Missing {path} -- run build_dpo_pairs_v2.py and upload its output."
with open(MANIFEST_PATH, encoding="utf-8") as f:
    manifest = json.load(f)

sft_data = sft_summary.get("data", {})
if (manifest["source"]["train_sha256"], manifest["source"]["validation_sha256"]) != \
        (sft_data.get("train_sha256"), sft_data.get("validation_sha256")):
    raise ValueError("The pairs were built from different train/validation files than SFT v2 was trained on. "
                     "Rebuild them from the exact redacted files SFT v2 used.")
if not manifest["source"].get("redacted"):
    raise ValueError("The manifest does not say the source data was redacted.")

pairs = {name: load_jsonl(path) for name, path in paths.items()}
if {n: len(p) for n, p in pairs.items()} != manifest["counts"]:
    raise ValueError(f"Pair counts {[len(p) for p in pairs.values()]} do not match the manifest {manifest['counts']}.")
for name, rows in pairs.items():
    for i, pair in enumerate(rows):
        messages = pair["prompt_messages"]
        if messages[0]["role"] != "system" or messages[-1]["role"] != "user":
            raise ValueError(f"{name} pair {i}: the prompt must start with 'system' and end with Contact's message.")
        if not pair["chosen"].strip() or not pair["rejected"].strip() or pair["chosen"] == pair["rejected"]:
            raise ValueError(f"{name} pair {i}: empty or identical chosen/rejected.")

prompt_key = lambda pair: json.dumps(pair["prompt_messages"], ensure_ascii=False)  # noqa: E731
train_keys = {prompt_key(p) for p in pairs["train"]}
assert not any(prompt_key(p) in train_keys for p in pairs["dev"] + pairs["test"]), "A dev/test prompt is also in train."
assert not {p["group"] for p in pairs["dev"]} & {p["group"] for p in pairs["test"]}, "A conversation is in both dev and test."

system_messages = {p["prompt_messages"][0]["content"] for p in sum(pairs.values(), [])}
assert len(system_messages) == 1, f"Expected one system message, found {len(system_messages)}."
SYSTEM_MESSAGE = system_messages.pop()

ARABIC = "؀-ۿ"
NAME_PREFIXES = "وبل"  # same rules as tools/private_names_lib.py


def load_private_names(path):
    with open(path, encoding="utf-8") as f:
        names = [line.strip() for line in f if line.strip() and not line.strip().startswith("#")]
    return sorted(set(names), key=len, reverse=True)


def compile_name_patterns(names):
    arabic = [n for n in names if re.search(f"[{ARABIC}]", n)]
    latin = [n for n in names if n not in arabic]
    ar = (re.compile(rf"(?<![{ARABIC}])(?P<pre>[{NAME_PREFIXES}])?(?P<name>{'|'.join(map(re.escape, arabic))})(?![{ARABIC}])")
          if arabic else None)
    la = re.compile(rf"(?<![A-Za-z])(?:{'|'.join(map(re.escape, latin))})(?![A-Za-z])", re.IGNORECASE) if latin else None
    return ar, la


def find_private_names(text, patterns):
    ar, la = patterns
    return ([m.group("name") for m in ar.finditer(text)] if ar else []) + ([m.group(0) for m in la.finditer(text)] if la else [])


NAME_PATTERNS = None
if os.path.exists(PRIVATE_NAMES_PATH):
    NAME_PATTERNS = compile_name_patterns(load_private_names(PRIVATE_NAMES_PATH))
    hits = Counter(name for name, rows in pairs.items() for p in rows
                   if any(find_private_names(t, NAME_PATTERNS)
                          for t in [m["content"] for m in p["prompt_messages"]] + [p["chosen"], p["rejected"]]))
    if hits:
        raise ValueError(f"Pairs containing a private name: {dict(hits)}. Rebuild the pairs from the redacted data.")
elif REQUIRE_NAME_CHECK:
    raise FileNotFoundError(f"{PRIVATE_NAMES_PATH} not found. Upload it, or set REQUIRE_NAME_CHECK = False.")

PAIRS_FINGERPRINT = {name: sha256_of(path) for name, path in [*paths.items(), ("manifest", MANIFEST_PATH)]}
print(f"Pairs: train {len(pairs['train'])} | dev {len(pairs['dev'])} | test {len(pairs['test'])} "
      f"| conversations in dev/test: {manifest['split']['dev_conversations']}/{manifest['split']['test_conversations']}")
print("Private-name scan:", "0 pairs with a listed name" if NAME_PATTERNS else "skipped")

## C. The run folder: resume, refuse, or skip

The run's identity (SFT model, pairs, `CONFIG`) is saved the first time. A later run with the same `RUN_NAME` must match it exactly. A finished run is not trained again.

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

run_config = json.loads(json.dumps({"base": BASE_IDENTITY, "pairs": PAIRS_FINGERPRINT, "config": CONFIG}))
os.makedirs(RUN_DIR, exist_ok=True)
if os.path.exists(RUN_CONFIG_PATH):
    with open(RUN_CONFIG_PATH, encoding="utf-8") as f:
        if json.load(f) != run_config:
            raise ValueError(f"{RUN_DIR} was started with a different SFT model, pairs or CONFIG. Use a new RUN_NAME.")
else:
    with open(RUN_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(run_config, f, indent=2)

TRAINING_COMPLETE = os.path.exists(RUN_SUMMARY_PATH) and os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json"))
LAST_CHECKPOINT = get_last_checkpoint(CHECKPOINT_DIR) if os.path.isdir(CHECKPOINT_DIR) else None
print("Already complete: training is skipped." if TRAINING_COMPLETE else
      f"Resuming from {LAST_CHECKPOINT}." if LAST_CHECKPOINT else "Starting a new run.")

free_gb = shutil.disk_usage(DRIVE_ROOT).free / 1e9
assert free_gb > MIN_FREE_DRIVE_GB, f"Only {free_gb:.2f} GB free on Drive: checkpoints would fail to save."

## D. Model and tokenizer

The tokenizer's end token is set to `<|im_end|>`, the token SFT v2 learned to write after every reply: TRL appends the tokenizer's end token to each reply, so this makes DPO's replies end exactly like SFT v2's. Section E checks it on the tokenized data.

The LoRA adapter is the same shape as SFT's. A fresh adapter changes nothing, so every measurement before training is a measurement of SFT v2 itself. TRL uses a frozen copy of this fresh adapter (named `ref`) as DPO's reference, which is therefore exactly SFT v2. Fingerprints and saving only ever use the trained adapter (`default`).

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(model_name=SFT_MODEL_PATH, max_seq_length=2048, load_in_4bit=False)

IM_END = "<|im_end|>"
IM_END_ID = tokenizer.convert_tokens_to_ids(IM_END)
assert IM_END_ID is not None and IM_END_ID != tokenizer.unk_token_id, "The tokenizer has no <|im_end|> token."
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.eos_token = IM_END
assert tokenizer.eos_token_id == IM_END_ID

model = FastLanguageModel.get_peft_model(
    model, r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"], random_state=CONFIG["seed"], use_gradient_checkpointing="unsloth",
)


def lora_fingerprint(m):
    # Only the trained adapter ("default"). TRL >= 1 adds a frozen copy named "ref" as the reference.
    return sum(p.detach().float().abs().sum().item() for n, p in m.named_parameters() if "lora_" in n and ".default." in n)


def adapter_file_fingerprint(path):
    from safetensors.torch import load_file
    state = load_file(path)
    return sum(t.float().abs().sum().item() for k, t in state.items() if "lora_" in k), state


def load_adapter_weights(adapter_path):
    # Loads LoRA weights into `model` and proves it by matching fingerprints.
    from peft import set_peft_model_state_dict
    path = adapter_path if adapter_path.endswith(".safetensors") else os.path.join(adapter_path, "adapter_model.safetensors")
    expected, state = adapter_file_fingerprint(path)
    set_peft_model_state_dict(model, state)
    assert abs(lora_fingerprint(model) - expected) <= 1e-3 * max(abs(expected), 1e-9), f"Adapter load failed: {path}"


FP_BEFORE = lora_fingerprint(model)
print("Loaded SFT v2 | <|im_end|> id:", IM_END_ID, "| pad:", repr(tokenizer.pad_token), "| LoRA fingerprint:", round(FP_BEFORE, 2))

## E. Format the pairs exactly like SFT v2

Each prompt is the SFT v2 model's chat template up to `<|im_start|>assistant
`; each reply is the text plus `<|im_end|>`. Pairs longer than `max_length` are dropped and counted. TRL would otherwise cut their end, which is the reply.

In [ ]:
def to_text_pair(pair):
    prompt = tokenizer.apply_chat_template(pair["prompt_messages"], tokenize=False, add_generation_prompt=True)
    return {"prompt": prompt, "chosen": pair["chosen"] + IM_END, "rejected": pair["rejected"] + IM_END}


def n_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def format_split(rows, name):
    kept, dropped = [], 0
    for pair in rows:
        text = to_text_pair(pair)
        prompt_ids = tokenizer(text["prompt"], add_special_tokens=False)["input_ids"]
        for side in ("chosen", "rejected"):
            full = tokenizer(text["prompt"] + text[side], add_special_tokens=False)["input_ids"]
            assert full[: len(prompt_ids)] == prompt_ids, f"{name}: the prompt tokens change when the reply is appended"
        if n_tokens(text["prompt"] + max(text["chosen"], text["rejected"], key=n_tokens)) > CONFIG["max_length"]:
            dropped += 1
            continue
        kept.append(text)
    if dropped / max(len(rows), 1) > MAX_DROPPED_FRACTION:
        raise ValueError(f"{name}: {dropped} of {len(rows)} pairs are over max_length={CONFIG['max_length']}.")
    print(f"{name}: {len(kept)} pairs ({dropped} dropped as too long)")
    return kept, dropped


train_text, train_dropped = format_split(pairs["train"], "train")
dev_text, dev_dropped = format_split(pairs["dev"], "dev")
test_text, test_dropped = format_split(pairs["test"], "test")
if TEST_PAIRS_LIMIT:
    test_text = test_text[:TEST_PAIRS_LIMIT]
train_ds, dev_ds = Dataset.from_list(train_text), Dataset.from_list(dev_text)

## F. Baseline: SFT v2, measured before any training

**Preference accuracy**: for each test pair, does the model give Ahmed's real reply a higher average per-token log-probability than the wrong one? Same tokens as SFT v2's labels (reply + `<|im_end|>`). Generations are greedy, so before/after differences come from the weights. The baseline is saved and reused when the run resumes.

In [ ]:
@torch.no_grad()
def completion_mean_logprob(prompt, completion):
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(prompt + completion, add_special_tokens=False)["input_ids"]
    ids = torch.tensor([full_ids], device=model.device)
    logits = model(input_ids=ids).logits[0, len(prompt_ids) - 1 : -1].float()
    token_logps = torch.log_softmax(logits, dim=-1).gather(1, ids[0, len(prompt_ids):, None]).squeeze(1)
    return token_logps.mean().item()


def preference_accuracy(rows):
    FastLanguageModel.for_inference(model)
    wins, margins = [], []
    for row in rows:
        chosen, rejected = completion_mean_logprob(row["prompt"], row["chosen"]), completion_mean_logprob(row["prompt"], row["rejected"])
        wins.append(chosen > rejected)
        margins.append(chosen - rejected)
    return {"accuracy": sum(wins) / len(wins), "mean_margin": float(statistics.mean(margins)),
            "n_pairs": len(wins), "per_pair_wins": wins}


@torch.no_grad()
def generate_reply(prompt):
    FastLanguageModel.for_inference(model)
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    output = model.generate(**encoded, max_new_tokens=100, do_sample=False, repetition_penalty=1.1,
                            no_repeat_ngram_size=4, eos_token_id=IM_END_ID, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(output[0, encoded["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def handwritten_prompt(text):
    return tokenizer.apply_chat_template([{"role": "system", "content": SYSTEM_MESSAGE}, {"role": "user", "content": text}],
                                         tokenize=False, add_generation_prompt=True)


def mcnemar(wins_a, wins_b):
    # Exact two-sided McNemar test on the same pairs scored twice: (b better, a better, p).
    assert len(wins_a) == len(wins_b), "Paired test needs the same pairs on both sides."
    fixed = sum((not a) and b for a, b in zip(wins_a, wins_b))
    broken = sum(a and (not b) for a, b in zip(wins_a, wins_b))
    n = fixed + broken
    p = min(1.0, 2 * sum(comb(n, i) for i in range(min(fixed, broken) + 1)) / 2 ** n) if n else 1.0
    return fixed, broken, p


def generations():
    return {"handwritten": [generate_reply(handwritten_prompt(m)) for m in HANDWRITTEN_TEST_MESSAGES],
            "heldout": [generate_reply(row["prompt"]) for row in test_text[:N_HELDOUT_GENERATIONS]]}


if TRAINING_COMPLETE:
    baseline = None  # read from the run summary in Section H
elif os.path.exists(BASELINE_PATH):
    with open(BASELINE_PATH, encoding="utf-8") as f:
        baseline = json.load(f)
    assert baseline["preference"]["n_pairs"] == len(test_text), "The saved baseline scored a different test set."
    print("Baseline reused from", BASELINE_PATH)
else:
    baseline = {"preference": preference_accuracy(test_text), "generations": generations()}
    with open(BASELINE_PATH, "w", encoding="utf-8") as f:
        json.dump(baseline, f, ensure_ascii=False)
if baseline:
    print(f"SFT v2 preference accuracy on {baseline['preference']['n_pairs']} test pairs: {baseline['preference']['accuracy']:.1%}")

## G. Train
### G1. The DPO configuration, with the anchor

The anchor is the SFT loss on `chosen` (v1's `rpo_alpha`). TRL ≥ 1 writes it as `loss_type=["sigmoid", "sft"]` with `loss_weights=[1, sft_weight]`; older TRL as `rpo_alpha`. Without either, the notebook stops (unless `ALLOW_PLAIN_DPO`). Any setting this TRL version does not know also stops it, instead of being dropped silently.

In [ ]:
from transformers import EarlyStoppingCallback
from trl import DPOConfig, DPOTrainer

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dpo_kwargs = dict(
    output_dir=CHECKPOINT_DIR,
    beta=CONFIG["beta"],
    max_length=CONFIG["max_length"],
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    optim="adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["eval_steps"],
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=CONFIG["seed"],
    data_seed=CONFIG["seed"],
)
# Unsloth replaces DPOConfig.__init__ with one that lists only some settings and passes the rest through **kwargs
# (loss_type among them), so the dataclass fields are the real list of accepted settings.
import dataclasses
accepted = ({field.name for field in dataclasses.fields(DPOConfig)} if dataclasses.is_dataclass(DPOConfig)
            else set(inspect.signature(DPOConfig.__init__).parameters))
if "loss_type" in accepted and "loss_weights" in accepted:
    dpo_kwargs.update(loss_type=["sigmoid", "sft"], loss_weights=[1.0, CONFIG["sft_weight"]])
    ANCHOR = f"loss_type=['sigmoid','sft'], loss_weights=[1.0, {CONFIG['sft_weight']}]"
elif "rpo_alpha" in accepted:
    dpo_kwargs["rpo_alpha"] = CONFIG["sft_weight"]
    ANCHOR = f"rpo_alpha={CONFIG['sft_weight']}"
elif ALLOW_PLAIN_DPO:
    ANCHOR = "NONE (plain DPO, allowed on purpose)"
else:
    raise RuntimeError(f"TRL {trl.__version__} supports neither loss_type/loss_weights nor rpo_alpha: the anchor "
                       "against likelihood displacement is unavailable. Set ALLOW_PLAIN_DPO = True only on purpose.")
unknown = [key for key in dpo_kwargs if key not in accepted]
if unknown:
    raise RuntimeError(f"TRL {trl.__version__}'s DPOConfig does not accept {unknown}. Update this cell; do not drop them silently.")
dpo_args = DPOConfig(**dpo_kwargs)
for key in ("beta", "loss_type", "loss_weights", "rpo_alpha"):  # the settings that define this run really arrived
    if key in dpo_kwargs:
        assert getattr(dpo_args, key, None) == dpo_kwargs[key], f"DPOConfig ignored {key}"
print("Anchor:", ANCHOR)

### G2. Build the trainer and check the tokenized replies

`ref_model=None`: with LoRA, the model with its adapter switched off is the frozen reference. The check below reads what TRL actually tokenized: every reply must end with exactly one `<|im_end|>`, and the prompt must be untouched.

In [ ]:
def make_trainer():
    common = dict(model=model, ref_model=None, args=dpo_args, train_dataset=train_ds, eval_dataset=dev_ds,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG["early_stopping_patience"])])
    try:
        return DPOTrainer(processing_class=tokenizer, **common)
    except TypeError:  # TRL before processing_class
        return DPOTrainer(tokenizer=tokenizer, **common)


trainer = make_trainer()

tokenized = trainer.train_dataset
completion_columns = [c for c in ("chosen_ids", "chosen_input_ids") if c in tokenized.column_names]
assert completion_columns, f"Unknown tokenized format (columns: {tokenized.column_names}); cannot verify the reply endings."
chosen_column = completion_columns[0]
rejected_column = chosen_column.replace("chosen", "rejected")
prompt_column = next(c for c in ("prompt_ids", "prompt_input_ids") if c in tokenized.column_names)
bad_endings = 0
for row in tokenized.select(range(min(200, len(tokenized)))):
    for column in (chosen_column, rejected_column):
        ids = row[column]
        if ids[-1] != IM_END_ID or ids.count(IM_END_ID) != 1:
            bad_endings += 1
assert bad_endings == 0, f"{bad_endings} tokenized replies do not end with exactly one <|im_end|>."
expected_prompt = tokenizer(train_ds[0]["prompt"], add_special_tokens=False)["input_ids"]
assert tokenized[0][prompt_column] == expected_prompt, "TRL changed the prompt tokens."
assert len(tokenized) == len(train_ds), "TRL dropped training pairs."
print(f"Checked: replies end with exactly one <|im_end|>; prompts unchanged; {len(tokenized)} training pairs.")

### G3. Train (resumes after a disconnect)

A step-0 evaluation records the starting point for the diagnosis. After training, the restored best checkpoint is verified against its file on disk, and the LoRA fingerprint must have changed (a run that trains nothing is caught).

In [ ]:
initial_eval = None
if TRAINING_COMPLETE:
    load_adapter_weights(ADAPTER_DIR)
    print("Already complete: loaded the saved adapter, no training.")
else:
    if LAST_CHECKPOINT is None:
        blockers = [cb for cb in trainer.callback_handler.callbacks
                    if type(cb).__name__ in ("NotebookProgressCallback", "EarlyStoppingCallback")]
        for cb in blockers:
            trainer.remove_callback(cb)
        try:
            initial_eval = trainer.evaluate()
        finally:
            for cb in blockers:
                trainer.add_callback(cb)
        with open(f"{RUN_DIR}/step0_eval.json", "w", encoding="utf-8") as f:
            json.dump(initial_eval, f)
    elif os.path.exists(f"{RUN_DIR}/step0_eval.json"):
        with open(f"{RUN_DIR}/step0_eval.json", encoding="utf-8") as f:
            initial_eval = json.load(f)

    FastLanguageModel.for_training(model)
    model.config.use_cache = False
    trainer.train(resume_from_checkpoint=LAST_CHECKPOINT)

    best = trainer.state.best_model_checkpoint
    best_file = os.path.join(best, "adapter_model.safetensors") if best else None
    if best_file and os.path.exists(best_file):
        expected, _ = adapter_file_fingerprint(best_file)
        if abs(lora_fingerprint(model) - expected) > 1e-3 * max(abs(expected), 1e-9):
            load_adapter_weights(best_file)
        print("Verified: the model holds", os.path.basename(best))
    change = abs(lora_fingerprint(model) - FP_BEFORE) / max(FP_BEFORE, 1e-9)
    assert change > MIN_RELATIVE_LORA_CHANGE, "The LoRA weights did not change: training was a no-op."
    print(f"Stopped at step {trainer.state.global_step} of {trainer.state.max_steps}; best {trainer.state.best_metric:.4f} "
          f"at {os.path.basename(str(best))}; LoRA change {change:.4%}")

## H. Evaluate and diagnose

The same test pairs, the same scoring and the same greedy decoding as the baseline. McNemar's test says whether the change is more than luck. The diagnosis names the setting to change next (see the table in A1).

In [ ]:
if TRAINING_COMPLETE:
    with open(RUN_SUMMARY_PATH, encoding="utf-8") as f:
        summary = json.load(f)
    after_pref, baseline_pref = summary["test_preference"]["dpo"], summary["test_preference"]["sft_v2"]
    print("Loaded the finished run's results.")
else:
    after_pref = preference_accuracy(test_text)
    baseline_pref = baseline["preference"]
    after_generations = generations()

fixed, broken, p_value = mcnemar(baseline_pref["per_pair_wins"], after_pref["per_pair_wins"])
delta = after_pref["accuracy"] - baseline_pref["accuracy"]
print(f"Test preference accuracy: SFT v2 {baseline_pref['accuracy']:.1%} -> DPO {after_pref['accuracy']:.1%} "
      f"({delta:+.1%}; {fixed} pairs fixed, {broken} broken, p={p_value:.4f}, n={after_pref['n_pairs']})")

In [ ]:
findings = []


def note(level, title, detail):
    findings.append({"level": level, "title": title, "detail": detail})


if not TRAINING_COMPLETE:
    evals = [h for h in trainer.state.log_history if "eval_loss" in h]
    start = initial_eval if initial_eval and "eval_logps/chosen" in initial_eval else (evals[0] if evals else None)
    if evals and start:
        best_eval = min(evals, key=lambda h: h["eval_loss"])
        first, last = start.get("eval_logps/chosen"), best_eval.get("eval_logps/chosen")
        if first is not None and last is not None:
            relative = (last - first) / abs(first)
            note("WARN" if relative < -0.05 else "OK", "Real replies " + ("became less likely" if relative < -0.05 else "stayed likely"),
                 f"eval logps/chosen {first:.2f} -> {last:.2f} ({relative:+.1%})" + (". Raise sft_weight." if relative < -0.05 else "."))
        early = [h.get("eval_rewards/accuracies") for h in evals if h["step"] <= 0.25 * evals[-1]["step"]]
        if any(a is not None and a >= 0.98 for a in early):
            note("WARN", "Pairs may be too easy", "Reward accuracy reached 98% early. Rebuild with --max-similarity 0.4.")
        if initial_eval and best_eval["eval_loss"] >= initial_eval["eval_loss"]:
            note("WARN", "Training never beat the starting point", "Lower learning_rate.")
        elif best_eval["step"] == evals[0]["step"]:
            note("WARN", "Best checkpoint was the first evaluation", "Lower learning_rate.")

if p_value < 0.05:
    note("OK" if delta > 0 else "WARN", f"Preference accuracy went {'UP' if delta > 0 else 'DOWN'} (significant)",
         f"{delta:+.1%}, p={p_value:.4f}.")
elif delta > 0:
    note("INFO", "Improvement within noise", f"{delta:+.1%}, p={p_value:.4f}. Lower beta in a new run.")
else:
    note("WARN", "No measurable improvement", f"{delta:+.1%}, p={p_value:.4f}. Lower beta, or check the pairs.")

if NAME_PATTERNS and not TRAINING_COMPLETE:
    name_count = lambda texts: sum(bool(find_private_names(t, NAME_PATTERNS)) for t in texts)  # noqa: E731
    before_texts = baseline["generations"]["handwritten"] + baseline["generations"]["heldout"]
    after_texts = after_generations["handwritten"] + after_generations["heldout"]
    note("OK" if name_count(after_texts) == 0 else "WARN", "Private names in generated replies",
         f"SFT v2: {name_count(before_texts)}, DPO: {name_count(after_texts)} of {len(after_texts)} (counted, not printed)")

for finding in findings:
    print(f"[{finding['level']}] {finding['title']}\n       {finding['detail']}")

## I. Save, compare, merge
### I1. Adapter, summary, private generations

`run_summary.json` holds totals only and can be shared. `generations_private.json` holds real text and must not be.

In [ ]:
if not TRAINING_COMPLETE:
    model.save_pretrained(ADAPTER_DIR, selected_adapters=["default"])  # not TRL's frozen "ref" copy
    tokenizer.save_pretrained(ADAPTER_DIR)
    weights = os.path.join(ADAPTER_DIR, "adapter_model.safetensors")
    assert os.path.exists(weights) and os.path.getsize(weights) > MIN_ADAPTER_BYTES, "Adapter save looks incomplete."

    summary = {
        "run_name": RUN_NAME,
        **run_config,
        "anchor": ANCHOR,
        "library_versions": LIBRARY_VERSIONS,
        "pairs_used": {"train": len(train_text), "dev": len(dev_text), "test": len(test_text),
                       "dropped_too_long": {"train": train_dropped, "dev": dev_dropped, "test": test_dropped}},
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_eval_loss": trainer.state.best_metric,
        "stopped_at_step": trainer.state.global_step,
        "planned_steps": trainer.state.max_steps,
        "step0_eval": initial_eval,
        "eval_history": [{k: v for k, v in h.items() if k.startswith("eval_") or k == "step"}
                         for h in trainer.state.log_history if "eval_loss" in h],
        "lora_fingerprint": {"before": FP_BEFORE, "after": lora_fingerprint(model)},
        "test_preference": {"sft_v2": baseline_pref, "dpo": after_pref,
                            "mcnemar": {"fixed": fixed, "broken": broken, "p": p_value}},
        "diagnosis": findings,
    }
    with open(RUN_SUMMARY_PATH, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    with open(GENERATIONS_PATH, "w", encoding="utf-8") as f:
        json.dump({"messages": HANDWRITTEN_TEST_MESSAGES, "sft_v2": baseline["generations"], "dpo": after_generations},
                  f, indent=2, ensure_ascii=False)
    print("Saved:", ADAPTER_DIR, "|", RUN_SUMMARY_PATH, "|", GENERATIONS_PATH, "(private)")

for i, message in enumerate(HANDWRITTEN_TEST_MESSAGES if not TRAINING_COMPLETE else []):
    print(f"\nUSER: {message}\nSFT v2: {baseline['generations']['handwritten'][i]}\nDPO:    {after_generations['handwritten'][i]}")

### I2. Head-to-head with other v2 runs

Only runs trained on **the same SFT model** (same `base` in their `run_config.json`) are scored on this run's test pairs and compared pair by pair. v1 runs (v7, v8) sit on a different base model: compare them in the evaluation notebook, not here.

In [ ]:
head_to_head = []
for other in COMPARE_WITH_RUNS:
    other_dir = f"{RUNS_DIR}/{other}"
    other_config_path = f"{other_dir}/run_config.json"
    if other == RUN_NAME or not os.path.exists(f"{other_dir}/adapter/adapter_model.safetensors"):
        print(f"Skipping {other}: no saved adapter.")
        continue
    with open(other_config_path, encoding="utf-8") as f:
        if json.load(f)["base"] != run_config["base"]:
            print(f"Skipping {other}: trained on a different SFT model.")
            continue
    load_adapter_weights(f"{other_dir}/adapter")
    other_pref = preference_accuracy(test_text)
    this_better, other_better, p_h2h = mcnemar(other_pref["per_pair_wins"], after_pref["per_pair_wins"])
    head_to_head.append({"other": other, "other_accuracy": other_pref["accuracy"], "this_accuracy": after_pref["accuracy"],
                         "this_better": this_better, "other_better": other_better, "p": p_h2h})
    print(f"{RUN_NAME} {after_pref['accuracy']:.1%} vs {other} {other_pref['accuracy']:.1%}: p={p_h2h:.4f}")
if head_to_head:
    load_adapter_weights(ADAPTER_DIR)
    summary["head_to_head"] = head_to_head
    with open(RUN_SUMMARY_PATH, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"{len(head_to_head)} head-to-head comparisons.")

### I3. Optional: merge into a standalone model

Set `MERGE_AFTER_TRAINING = True` in A1 to write `merged/` (~6–7 GB). The evaluation notebook can also load the adapter on top of SFT v2 directly, without merging.

In [ ]:
if MERGE_AFTER_TRAINING:
    assert shutil.disk_usage(DRIVE_ROOT).free / 1e9 > 7, "A merged 3B model needs ~7 GB free."
    for name in ("trainer", "model"):
        globals().pop(name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    merge_model, merge_tokenizer = FastLanguageModel.from_pretrained(model_name=ADAPTER_DIR, max_seq_length=2048, load_in_4bit=False)
    merge_model.save_pretrained_merged(MERGED_DIR, merge_tokenizer, save_method="merged_16bit")
    shutil.copy(RUN_SUMMARY_PATH, os.path.join(MERGED_DIR, "dpo_run_summary.json"))
    print("Merged:", MERGED_DIR)
else:
    print("Merge skipped (MERGE_AFTER_TRAINING = False).")

## Known limitations
- **About 1,000+ test pairs** (all validation conversations not in dev) make a ~3-point difference measurable; smaller ones may still read as noise.
- **Dev and test come from validation**, split by conversation: no shared text, but the same person and period.
- **Training pairs come from SFT v2's training data**, as in v1: DPO refines replies the SFT model already learned.
- **The anchor and the tokenization follow TRL's current API**; the cells stop with a clear message if a future TRL changes it, rather than continuing silently.